## Frozen-feature screening

Screens pretrained backbones and input variants with logistic-regression heads on the identity-cluster folds of `manifest.csv` (from `tongue-dataprep`, cv mode). Originals only; augmentations are not used.

- **Embeddings**: one per (backbone, input variant, size), mean over 10 views (5 crops at 90% x horizontal flip); cached in `features/`.
- **Head**: StandardScaler + LogisticRegression; `C` chosen by 3-fold group CV over training clusters; threshold by the same rule as the training notebook (recall s.t. precision >= 0.90, ties to the middle).
- **Order**: (1) backbones on `circle`; (2) inputs `full`, `circle`, `seg`, `segcrop` on the best two; (3) gray-world white balance; (4) 224 vs 448; (5) ensemble of the two finalists. Every finalist is also scored on `corners` (lips, clothing, background only).
- **Fresh folds**: the winner is also scored on `fold_s2` (StratifiedGroupKFold, seed 2024), which screening never looks at.
- **Outputs**: `masks/<family>.png`, `screen_oof.csv`, `screen_summary.csv`, `screen_choice.json`.
- `SMOKE=1` (local CPU check): 100 originals, ResNet-50 and DINOv2-S without pretrained weights, 2 views, size 224 only.

In [ ]:
import os, re, json, random
from pathlib import Path
import numpy as np
import pandas as pd
import cv2
import torch
import timm
import torchvision
from torchvision import transforms
from torchvision.models import resnet50, ResNet50_Weights
from PIL import Image
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, precision_score, recall_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

MANIFEST = Path(os.environ.get('MANIFEST', '/kaggle/input/notebooks/edamhakim/tongue-dataprep/manifest.csv'))
OUT = Path(os.environ.get('OUTPUT_BASE', '/kaggle/working'))
SMOKE = os.environ.get('SMOKE') == '1'
SEED, N_FOLDS = 42, 5
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
(OUT / 'features').mkdir(parents=True, exist_ok=True)
(OUT / 'masks').mkdir(parents=True, exist_ok=True)
print('Device:', device, '| torch', torch.__version__, '| timm', timm.__version__, '| smoke', SMOKE)

In [ ]:
# Tongue masks and white balance (ported verbatim from analysis/tongue_masks.py, eye-checked on 40 originals)
MIN_AREA, MAX_AREA, WORK = 0.20, 0.95, 300


def circle_mask(h, w):
    yy, xx = np.mgrid[-1:1:h * 1j, -1:1:w * 1j]
    return ((xx ** 2 + yy ** 2) <= 0.85 ** 2).astype(np.uint8)


def gray_world(img_bgr):
    img = img_bgr.astype(np.float32)
    means = img.reshape(-1, 3).mean(0)
    return np.clip(img * (means.mean() / np.maximum(means, 1e-6)), 0, 255).round().astype(np.uint8)


def tongue_mask(img_bgr):
    """(uint8 0/1 mask at image size, fell_back)."""
    h, w = img_bgr.shape[:2]
    small = cv2.resize(img_bgr, (WORK, WORK))
    yy, xx = np.mgrid[-1:1:WORK * 1j, -1:1:WORK * 1j]
    gc = np.full((WORK, WORK), cv2.GC_PR_BGD, np.uint8)
    # Crops are tight and centred: position seeds work for coated, pale and purple tongues alike
    gc[(xx / 0.8) ** 2 + (yy / 0.9) ** 2 <= 1] = cv2.GC_PR_FGD
    gc[(xx / 0.4) ** 2 + (yy / 0.45) ** 2 <= 1] = cv2.GC_FGD
    b = int(0.03 * WORK)
    gc[:b], gc[-b:], gc[:, :b], gc[:, -b:] = cv2.GC_BGD, cv2.GC_BGD, cv2.GC_BGD, cv2.GC_BGD
    fg = np.zeros((WORK, WORK), np.uint8)
    if (gc == cv2.GC_PR_FGD).any():
        try:
            cv2.grabCut(small, gc, None, np.zeros((1, 65)), np.zeros((1, 65)), 5, cv2.GC_INIT_WITH_MASK)
            fg = ((gc == cv2.GC_FGD) | (gc == cv2.GC_PR_FGD)).astype(np.uint8)
        except cv2.error:
            pass  # degenerate colour models: treated like an empty mask, i.e. fallback
    n, lab, stats, _ = cv2.connectedComponentsWithStats(fg)
    if n > 1:
        fg = (lab == 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])).astype(np.uint8)
        holes = fg.copy()
        cv2.floodFill(holes, np.zeros((WORK + 2, WORK + 2), np.uint8), (0, 0), 1)
        fg = fg | (1 - holes)
    if not MIN_AREA <= fg.mean() <= MAX_AREA:
        return circle_mask(h, w), True
    return cv2.resize(fg, (w, h), interpolation=cv2.INTER_NEAREST), False


def fresh_folds(orig):
    """Fold 0-4 per original from StratifiedGroupKFold(5, shuffle, 2024) on label grouped by group_cluster."""
    folds = pd.Series(-1, index=orig.index)
    sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=2024)
    for k, (_, te) in enumerate(sgkf.split(orig, orig['label'], orig['group_cluster'])):
        folds.iloc[te] = k
    return folds

In [ ]:
root = MANIFEST.parent
orig = pd.read_csv(MANIFEST).query('is_orig == 1').reset_index(drop=True)
orig['fold_s2'] = fresh_folds(orig)
assert orig.groupby('group_cluster').fold_s2.nunique().max() == 1
if SMOKE:  # 10 per class per cluster fold, so every fold keeps both classes
    orig = orig.groupby(['fold_cluster', 'label'], group_keys=False).apply(lambda g: g.head(10)).reset_index(drop=True)
print(f'{len(orig)} originals, {orig.group_cluster.nunique()} identity clusters')

fell_back = 0
for fam, f in zip(orig.family, orig.file):
    p = OUT / 'masks' / f'{fam}.png'
    if not p.exists():
        m, fb = tongue_mask(cv2.imread(str(root / f)))
        cv2.imwrite(str(p), m * 255)
        fell_back += fb
print('masks written; fallback to circle:', fell_back)

In [ ]:
MEAN_RGB = np.array([124, 116, 104], np.uint8)  # ImageNet mean colour = 0 after normalization


def prepare(file, family, variant):
    """RGB PIL image for one input variant: full, circle, seg, segcrop, corners; '+gw' = gray-world first."""
    base, gw = variant.split('+')[0], variant.endswith('+gw')
    bgr = cv2.imread(str(root / file))
    if gw:
        bgr = gray_world(bgr)
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    h, w = rgb.shape[:2]
    if base == 'full':
        keep = np.ones((h, w), np.uint8)
    elif base == 'circle':
        keep = circle_mask(h, w)
    elif base in ('seg', 'segcrop'):
        keep = (cv2.imread(str(OUT / 'masks' / f'{family}.png'), 0) > 0).astype(np.uint8)
    elif base == 'corners':  # outside the inscribed circle, as in the training notebook's periphery probe
        yy, xx = np.mgrid[-1:1:h * 1j, -1:1:w * 1j]
        keep = ((xx ** 2 + yy ** 2) >= 1.0).astype(np.uint8)
    else:
        raise ValueError(variant)
    rgb = np.where(keep[..., None] == 1, rgb, MEAN_RGB)
    if base == 'segcrop':
        ys, xs = np.where(keep)
        my, mx = int(0.05 * h), int(0.05 * w)
        rgb = rgb[max(0, ys.min() - my):ys.max() + my + 1, max(0, xs.min() - mx):xs.max() + mx + 1]
    return Image.fromarray(rgb.astype(np.uint8))

In [ ]:
BACKBONES = {'dinov2_s': 'vit_small_patch14_dinov2.lvd142m', 'dinov2_b': 'vit_base_patch14_dinov2.lvd142m',
             'convnext_s': 'convnext_small.fb_in22k_ft_in1k', 'effv2_s': 'tf_efficientnetv2_s.in21k_ft_in1k',
             'resnet50': 'torchvision'}
N_VIEWS = 2 if SMOKE else 10
_models = {}


def load_backbone(name):
    if name not in _models:
        if name == 'resnet50':
            m = resnet50(weights=None if SMOKE else ResNet50_Weights.IMAGENET1K_V2)
            m.fc = torch.nn.Identity()
            mean, std = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
        else:
            kw = {'dynamic_img_size': True} if name.startswith('dinov2') else {}
            m = timm.create_model(BACKBONES[name], pretrained=not SMOKE, num_classes=0, **kw)
            cfg = timm.data.resolve_model_data_config(m)
            mean, std = cfg['mean'], cfg['std']
        _models.clear()  # keep one backbone in memory
        _models[name] = (m.eval().to(device), transforms.Normalize(mean, std))
    return _models[name]


def views(img, size):
    """N_VIEWS tensors: 5 crops at 90% (centre + corners) x horizontal flip; smoke = centre x flip."""
    big = transforms.Resize((round(size / 0.9), round(size / 0.9)), interpolation=transforms.InterpolationMode.BICUBIC)(img)
    crops = transforms.FiveCrop(size)(big)
    crops = [crops[4]] if SMOKE else list(crops)
    out = []
    for c in crops:
        t = transforms.ToTensor()(c)
        out += [t, torch.flip(t, dims=[2])]
    return out


@torch.no_grad()
def extract(backbone, variant, size):
    """(n_originals, n_features) mean embedding over views; cached per (backbone, variant, size)."""
    path = OUT / 'features' / f'{backbone}__{variant}__{size}{"_smoke" if SMOKE else ""}.npy'
    if path.exists():
        return np.load(path)
    model, norm = load_backbone(backbone)
    feats = []
    for file, fam in zip(orig.file, orig.family):
        x = torch.stack([norm(v) for v in views(prepare(file, fam, variant), size)]).to(device)
        feats.append(model(x).float().mean(0).cpu().numpy())
    X = np.stack(feats)
    np.save(path, X)
    return X


C_GRID = (1e-3, 1e-2, 1e-1, 1.0, 10.0)


def choose_threshold(labels, probs):
    """Maximize recall subject to precision >= 0.90; ties go to the middle of the tied thresholds."""
    thresholds = np.round(np.arange(0.10, 0.901, 0.01), 2)
    scores = np.array([(precision_score(labels, probs >= t, zero_division=0) >= 0.90) * recall_score(labels, probs >= t, zero_division=0)
                       for t in thresholds])
    if scores.max() == 0:
        return 0.50
    tied = np.flatnonzero(scores == scores.max())
    return float(thresholds[tied[len(tied) // 2]])


def make_clf(C):
    return make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=5000))


def fit_head(X_tr, y_tr, g_tr):
    """Pick C by 3-fold group CV over training clusters (ROC-AUC); threshold from those inner predictions."""
    inner = list(StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=SEED).split(X_tr, y_tr, g_tr))
    best = (-1.0, None, None)
    for C in C_GRID:
        oof, aucs = np.full(len(y_tr), np.nan), []
        for tr, va in inner:
            if len(np.unique(y_tr[tr])) < 2:
                continue  # an inner training split with one class cannot be fit
            oof[va] = make_clf(C).fit(X_tr[tr], y_tr[tr]).predict_proba(X_tr[va])[:, 1]
            if len(np.unique(y_tr[va])) == 2:
                aucs.append(roc_auc_score(y_tr[va], oof[va]))
        score = np.mean(aucs) if aucs else 0.5
        if score > best[0]:
            best = (score, C, oof)
    _, C, oof = best
    ok = ~np.isnan(oof)
    threshold = choose_threshold(y_tr[ok], oof[ok]) if ok.any() and len(np.unique(y_tr[ok])) == 2 else 0.5
    return make_clf(C).fit(X_tr, y_tr), threshold


def run_screen(keys, fold_col='fold_cluster'):
    """OOF rows for one feature set; keys = list of (backbone, variant, size), concatenated for ensembles."""
    X = np.hstack([extract(*k) for k in keys])
    y, g = orig.label.values, orig.group_cluster.values
    rows = []
    for fold in sorted(orig[fold_col].unique()):
        te = orig[fold_col].values == fold
        clf, threshold = fit_head(X[~te], y[~te], g[~te])
        rows.append(pd.DataFrame({'scheme': 'cluster', 'fold_col': fold_col, 'model': '+'.join(k[0] for k in keys),
                                  'input': '+'.join(f'{k[1]}@{k[2]}' for k in keys), 'fold': fold,
                                  'file': orig.file[te].values, 'label': y[te], 'prob': clf.predict_proba(X[te])[:, 1],
                                  'threshold': threshold, 'group_cluster': g[te]}))
    return pd.concat(rows, ignore_index=True)

In [ ]:
def _cluster_draws(df, B, seed):
    rng = np.random.default_rng(seed)
    groups = {c: [g.index.values for _, g in df[df.label == c].groupby('group_cluster')] for c in (0, 1)}
    for _ in range(B):
        yield np.concatenate([groups[c][j] for c in (0, 1) for j in rng.integers(0, len(groups[c]), len(groups[c]))])


def cluster_bootstrap_auc(df, B=2000, seed=42):
    df = df.reset_index(drop=True)
    aucs = [roc_auc_score(df.label.values[i], df.prob.values[i]) for i in _cluster_draws(df, B, seed)]
    return (float(roc_auc_score(df.label, df.prob)), *np.percentile(aucs, [2.5, 97.5]).tolist())


def paired_delta_auc(df_a, df_b, B=2000, seed=42):
    both = df_a.merge(df_b[['file', 'prob']], on='file', suffixes=('', '_b'), validate='one_to_one').reset_index(drop=True)
    d = lambda i: roc_auc_score(both.label.values[i], both.prob.values[i]) - roc_auc_score(both.label.values[i], both.prob_b.values[i])
    deltas = [d(i) for i in _cluster_draws(both, B, seed)]
    return (float(d(np.arange(len(both)))), *np.percentile(deltas, [2.5, 97.5]).tolist())


def summarize(run):
    auc, lo, hi = cluster_bootstrap_auc(run, B=300 if SMOKE else 2000)
    person = run.groupby('group_cluster').agg(label=('label', 'first'), prob=('prob', 'mean'))
    pred = run.prob >= run.threshold
    return {'model': run.model.iat[0], 'input': run.input.iat[0], 'fold_col': run.fold_col.iat[0],
            'auc_pooled': round(auc, 3), 'auc_lo': round(lo, 3), 'auc_hi': round(hi, 3),
            'auc_fold_mean': round(run.groupby('fold').apply(lambda g: roc_auc_score(g.label, g.prob)).mean(), 3),
            'auc_person': round(roc_auc_score(person.label, person.prob), 3),
            'sens': round(float(pred[run.label == 1].mean()), 3), 'spec': round(float((~pred)[run.label == 0].mean()), 3)}

In [ ]:
# Self-checks: single-class inner split does not crash the head; DINOv2 output size at 224 and 448; TTA = mean of views
_X = np.random.default_rng(0).normal(size=(12, 4)); _y = np.array([0] * 6 + [1] * 6); _g = np.repeat(np.arange(4), 3)
_clf, _t = fit_head(_X, _y, _g)
assert 0.1 <= _t <= 0.9
_m, _norm = load_backbone('dinov2_s')
with torch.no_grad():
    for _s in (224, 448):
        assert _m(torch.zeros(1, 3, _s, _s, device=device)).shape == (1, 384), _s
_one = orig.iloc[:1]
_img = prepare(_one.file.iat[0], _one.family.iat[0], 'circle')
with torch.no_grad():
    _manual = _m(torch.stack([_norm(v) for v in views(_img, 224)]).to(device)).float().mean(0).cpu().numpy()
_saved, orig = orig, _one
try:
    _cached = OUT / 'features' / f'dinov2_s__circle__224{"_smoke" if SMOKE else ""}.npy'
    _cached.unlink(missing_ok=True)
    assert np.allclose(extract('dinov2_s', 'circle', 224)[0], _manual, atol=1e-4)
finally:
    orig = _saved
    _cached.unlink(missing_ok=True)
print('self-checks ok')

In [ ]:
results, summary = {}, []


def score(keys, fold_col='fold_cluster'):
    key = (tuple(keys), fold_col)
    if key not in results:
        results[key] = run_screen(list(keys), fold_col)
        summary.append(summarize(results[key]))
        print(f"  {summary[-1]['model']:<20} {summary[-1]['input']:<28} {fold_col:<12} AUC {summary[-1]['auc_pooled']:.3f} "
              f"[{summary[-1]['auc_lo']:.3f}, {summary[-1]['auc_hi']:.3f}]  person {summary[-1]['auc_person']:.3f}")
    return float(roc_auc_score(results[key].label, results[key].prob))


backbones = ['resnet50', 'dinov2_s'] if SMOKE else list(BACKBONES)
sizes = [224] if SMOKE else [224, 448]

print('Step 1: backbones on circle @224')
step1 = {b: score([(b, 'circle', 224)]) for b in backbones}
top2 = sorted(step1, key=step1.get, reverse=True)[:2]
best = {b: (b, 'circle', 224) for b in top2}

print('Step 2: input variants')
for b in top2:
    for v in ('full', 'seg', 'segcrop'):
        if score([(b, v, 224)]) > score([best[b]]):
            best[b] = (b, v, 224)

print('Step 3: gray-world white balance')
for b in top2:
    gw = (b, best[b][1] + '+gw', best[b][2])
    if score([gw]) > score([best[b]]):
        best[b] = gw

print('Step 4: resolution')
for b in top2:
    for s in sizes[1:]:
        hi = (b, best[b][1], s)
        if score([hi]) > score([best[b]]):
            best[b] = hi

print('Step 5: ensemble of the two finalists')
finalists = {f'{b}': [best[b]] for b in top2}
finalists['ensemble'] = [best[b] for b in top2]
final_auc = {name: score(keys) for name, keys in finalists.items()}
winner = max(final_auc, key=final_auc.get)

print('Corners-only check of the finalists')
for name, keys in finalists.items():
    score([(b, 'corners' + ('+gw' if v.endswith('+gw') else ''), s) for b, v, s in keys])

print('Fresh folds (fold_s2) for the winner and the frozen ResNet-50 circle baseline')
score(finalists[winner], 'fold_s2')
score([('resnet50', 'circle', 224)], 'fold_s2')
win_s2 = results[(tuple(finalists[winner]), 'fold_s2')]
base_s2 = results[((('resnet50', 'circle', 224),), 'fold_s2')]
d, lo, hi = paired_delta_auc(win_s2, base_s2, B=300 if SMOKE else 2000)
print(f'Winner {winner} vs frozen ResNet-50 circle on fold_s2: delta AUC {d:+.3f} [{lo:+.3f}, {hi:+.3f}]')

pd.concat(results.values(), ignore_index=True).to_csv(OUT / 'screen_oof.csv', index=False)
summary_df = pd.DataFrame(summary)
summary_df.to_csv(OUT / 'screen_summary.csv', index=False)
json.dump({'winner': winner, 'finalists': {k: [list(x) for x in v] for k, v in finalists.items()},
           'delta_vs_resnet50_circle_fold_s2': [round(d, 3), round(lo, 3), round(hi, 3)]},
          open(OUT / 'screen_choice.json', 'w'), indent=2)
print(summary_df.sort_values(['fold_col', 'auc_pooled'], ascending=[True, False]).to_string(index=False))